In [ ]:
!unzip Image.zip

In [ ]:
from skimage import color, data, exposure, morphology, measure
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from skimage.filters import try_all_threshold, threshold_otsu, threshold_local, sobel, gaussian
from skimage.color import rgb2gray, label2rgb
from skimage.transform import rotate, rescale, resize
from skimage.util import random_noise
from skimage.restoration import denoise_tv_chambolle, denoise_bilateral, inpaint
from skimage.segmentation import slic
from skimage.measure import find_contours
from skimage.feature import canny, corner_harris, Cascade
from matplotlib import patches

def crop_face(result, detected, title="Face detected"):
    for d in detected:
        print(d)
        rostro= result[d['r']:d['r']+d['width'], d['c']:d['c']+d['height']]

        plt.figure(figsize=(8, 6))
        plt.imshow(rostro)
        plt.title(title)
        plt.axis('off')
        plt.show()

def show_detected_face(result, detected, title="Face image"):
    plt.figure()
    plt.imshow(result)
    img_desc = plt.gca()
    plt.set_cmap('gray')
    plt.title(title)
    plt.axis('off')

    for patch in detected:

        img_desc.add_patch(
            patches.Rectangle(
                (patch['c'], patch['r']),
                patch['width'],
                patch['height'],
                fill=False,
                color='r',
                linewidth=2)
        )
    plt.show()
    crop_face(result, detected)

def show_image(image, title="Image", cmap_type='gray'):
    plt.imshow(image, cmap=cmap_type)
    plt.title(title)
    plt.axis('off')
    plt.show()

def plot_comparison(original, filtered, title_filtered):
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(8, 6), sharex=True, sharey=True)
    ax1.imshow(original, cmap=plt.cm.gray)
    ax1.set_title('original')
    ax1.axis('off')
    ax2.imshow(filtered, cmap=plt.cm.gray)
    ax2.set_title(title_filtered)
    ax2.axis('off')

def show_image_contour(image, contours):
    plt.figure()
    for n, contour in enumerate(contours):
        plt.plot(contour[:, 1], contour[:, 0], linewidth=3)
    plt.imshow(image, interpolation='nearest', cmap='gray_r')
    plt.title('Contours')
    plt.axis('off')
    plt.show()

def show_image_with_corners(image, coords, title="Corners detected"):
    plt.figure(figsize=(12, 10))
    plt.imshow(image, interpolation='nearest', cmap='gray')
    plt.title(title)
    plt.plot(coords[:, 1], coords[:, 0], '+r', markersize=35)
    plt.axis('off')
    plt.show()
    plt.close()

# **Scikit-Image**

In [ ]:
from skimage import data

rocket_image = data.rocket()
rocket_image

## **RGB vs Grayscale**

In [ ]:
from skimage import color
import matplotlib.pyplot as plt


grayscale = color.rgb2gray(rocket_image)
plt.imshow(grayscale, cmap='gray')
plt.show()


In [ ]:
rgb = color.gray2rgb(grayscale)
plt.imshow(rgb)

## **Visualizing images function**

In [ ]:
def show_image(image, title="Image", cmap_type='gray'):
    plt.imshow(image, cmap=cmap_type)
    plt.title(title)
    plt.axis('off')
    plt.show()

In [ ]:
show_image(grayscale, "Grayscale")

# **Image as NdArrays**

In [ ]:
image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/sevilleup(2).jpg")
type(image)

red = image[:, :, 0]
green = image[:, :, 1]
blue = image[:, :, 2]

In [ ]:
show_image(image)

In [ ]:
plt.imshow(image)
plt.title('image')
plt.axis('off')
plt.show()

## **Colors with NumPy**

In [ ]:
plt.imshow(red, cmap='gray')
plt.title('Red')
plt.axis('off')
plt.show()

In [ ]:
plt.imshow(blue, cmap='gray')
plt.title('Green')
plt.axis('off')
plt.show()

In [ ]:
plt.imshow(blue, cmap='gray')
plt.title('Blue')
plt.axis('off')
plt.show()

## **Shapes**

In [ ]:
image.shape

## **Sizes**

In [ ]:
image.size

## **Vertical Flip**

In [ ]:
vertical = np.flipud(image)
show_image(vertical, "Vertically Flipped Image")

## **Horizontal Flip**

In [ ]:
horizon = np.fliplr(image)
show_image(horizon)

## **Histogram of a picture?**

In [ ]:
red = image[:, :, 0]
plt.hist(red.ravel(), bins=256)
plt.show()

In [ ]:
blue = image[:, :, 2]

plt.hist(blue.ravel(), bins=256)
plt.show()

# **Thresholding**

In [ ]:
# Optimal threshold value

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/bw.jpg")


thresh = 127
binary = image > thresh
inverted_binary = image <= thresh
show_image(image, 'original')


In [ ]:
show_image(binary.astype(float), 'Threshold')

In [ ]:
show_image(inverted_binary.astype(float), 'Inverted Threshold')

## **More Thresholding Algorithms**

In [ ]:
from skimage.filters import try_all_threshold

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/bw.jpg")
image = rgb2gray(image)

fig, ax = try_all_threshold(image, verbose=False)
plt.show()

In [ ]:
from skimage.filters import threshold_otsu
from skimage.color import rgb2gray

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/bw.jpg")
image = rgb2gray(image)

thresh = threshold_otsu(image)
binary_global = image > thresh
show_image(binary_global, "Global Thresholding")

In [ ]:
from skimage.filters import threshold_local

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/bw.jpg")
block_size = 35
local_thresh = threshold_local(image, block_size, offset=10)
binary_local = image > local_thresh
show_image(binary_local.astype(float), "Local Thresholding")

Not being sure about what thresholding method to use isn't a problem. In fact, scikit-image provides us with a function to check multiple methods and see for ourselves what the best option is. It returns a figure comparing the outputs of different global thresholding methods.

In [ ]:
# Import the try all function
from skimage.filters import try_all_threshold

# Import the rgb to gray convertor function
from skimage.color import rgb2gray

fruits_image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/fruits-2.jpg")

# Turn the fruits_image to grayscale
grayscale = rgb2gray(fruits_image)

# Use the try all method on the resulting grayscale image
fig, ax = try_all_threshold(grayscale, verbose=False)

# Show the resulting plots
show_image(fruits_image, 'Original image')
plt.show()

What type of thresholding is best used to binarize an image?

In [ ]:
# Import threshold and gray convertor functions
from skimage.filters import threshold_otsu
from skimage.color import rgb2gray

tools_image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 1/shapes52.jpg")

# Turn the image grayscale
gray_tools_image = rgb2gray(tools_image)

# Obtain the optimal thresh
thresh = threshold_otsu(gray_tools_image)

# Obtain the binary image by applying thresholding
binary_image = gray_tools_image > thresh

# Show the resulting binary image
show_image(tools_image, 'Original image')
show_image(binary_image, 'Binarized image')

# **Edge Detection**

## **Sobel**

### **Comparing plots**

In [ ]:
def plot_comparison(original, filtered, title_filtered):
    fig, (ax1, ax2) = plt.subplots(ncols=2, figsize=(8, 6), sharex=True, sharey=True)
    ax1.imshow(original, cmap=plt.cm.gray)
    ax1.set_title('original')
    ax1.axis('off')
    ax2.imshow(filtered, cmap=plt.cm.gray)
    ax2.set_title(title_filtered)
    ax2.axis('off')

In [ ]:
from skimage.filters import sobel

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/image_cat.jpg")
image = rgb2gray(image) # Must do if the image is not grayscale

# Apply edge detection filter
edge_sobel = sobel(image)
plot_comparison(image, edge_sobel, "Edge with Sobel")

# **Gaussian Smoothing**

In [ ]:
from skimage.filters import gaussian

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/building_image.jpg")
gaussian_image = gaussian(image, channel_axis=-1) # multichannel เลิกใช้แล้ว
plot_comparison(image, gaussian_image, "Gaussian Filter")

In [ ]:
image = rgb2gray(image) # Must do if the image is not grayscale

# Apply edge detection filter
edge_sobel = sobel(image)
plot_comparison(image, edge_sobel, "Edge with Sobel")

# **Contrast Enhancement**

In [ ]:
from skimage import exposure

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/chest_xray_image.png")

# Obtain the equalized image
image_eq = exposure.equalize_hist(image)
plot_comparison(image, image_eq, "Histogram equalized")

## **Adaptive equalization**

In [ ]:
from skimage import exposure

image_adapt = exposure.equalize_adapthist(image, clip_limit=0.03)
plot_comparison(image, image_adapt, "Adaptive equalized")

## **Historam of Image**

In [ ]:
plt.title('Histogram of image')
plt.hist(image.ravel(), bins=256)
plt.show()

### **More examples**

In [ ]:
from skimage import exposure

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/image_aerial.tiff")

# Obtain the equalized image
image_eq = exposure.equalize_hist(image)
plot_comparison(image, image_eq, "Histogram equalized")

In [ ]:
# Import the necessary modules
from skimage import data, exposure

# Load the image
original_image = data.coffee()

# Apply the adaptive equalization on the original image
adapthist_eq_image = exposure.equalize_adapthist(original_image, clip_limit=0.03)

# Compare the original image to the equalized
plot_comparison(original_image, adapthist_eq_image, "Adaptive equalized")

# **Rotating**

In [ ]:
from skimage.transform import rotate

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/image_cat.jpg")

# Rotate 90 degree clockwise
image_rotated = rotate(image, -90)
plot_comparison(image, image_rotated, "Rotated")

image_rotated = rotate(image, 90)
plot_comparison(image, image_rotated, "Rotated")


## **Rescale**

In [ ]:
from skimage.transform import rescale

# 4 times smaller
image_rescaled = rescale(image, 1/4, anti_aliasing=True, channel_axis=-1) # มันไม่มี multichannel
show_image(image)
show_image(image_rescaled, "Rescaled")

## **Aliasing digital image**

In [ ]:
from skimage.transform import resize

height = 400
width = 500

image_resized = resize(image, (height, width), anti_aliasing=True)

show_image(image)
show_image(image_resized, "Resized")

## **Resize proportionally**

In [ ]:
from skimage.transform import resize

height = image.shape[0]/4
width = image.shape[1]/4

image_resized = resize(image, (height, width), anti_aliasing=True)

show_image(image)
show_image(image_resized, "Resized")

# **Morphology**

- better for binary images

คือการสร้าง strucuring element เข้าไปแทรกในรูปใหญ่ เพื่อดำเนินการอะไรสักอย่าง


In [ ]:
from skimage import morphology

square = morphology.square(4)
square

In [ ]:
morphology.rectangle(4,2)

## **Erosion in scikit-image**

In [ ]:
from skimage import morphology

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/world_image_binary.jpg")
selem = morphology.rectangle(12, 6)
eroded_image = morphology.binary_erosion(image, footprint=selem)
plot_comparison(image, eroded_image, "Eroded")

In [ ]:
# Import the morphology module
from skimage import morphology

upper_r_image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/r5.png")

# Obtain the eroded shape
eroded_image_shape = morphology.binary_erosion(upper_r_image)

# See results
show_image(upper_r_image, 'Original')
show_image(eroded_image_shape.astype(float), 'Eroded image') # ต้องมี .astype(float)

## **Dilation**

In [ ]:
from skimage import morphology

dilated_image = morphology.binary_dilation(image)
plot_comparison(image, dilated_image, "Dilated")

# **Image Restoration**

In [ ]:
defect_image = data.astronaut()
show_image(defect_image)

In [ ]:
defect_image.shape

In [ ]:
def get_mask(image):
    ''' Creates mask with four defect regions '''
    mask = np.zeros(image.shape[:-1])
    mask[20:60, 0:20] = 1
    mask[160:180, 70:155] = 1
    mask[30:90, 310:350] = 1
    mask[350:390, 70:90] = 1
    return mask

In [ ]:
from skimage.restoration import inpaint

def get_mask(image): # สร้างรอยดำบนภาพก่อน
    mask = np.ones(image.shape[:-1])
    mask[20:100, 215:280] = 0
    return mask

mask = get_mask(defect_image)
damaged_image = defect_image.copy()
damaged_image[mask == 0] = 0
show_image(damaged_image)

def get_mask(image): # อันนี้เป็น mask ที่จะใช้ restore ภาพ
    ''' Creates mask with four defect regions '''
    mask = np.zeros(image.shape[:-1])
    mask[20:100, 215:280] = 1
    return mask

mask = get_mask(damaged_image)
show_image(mask, title='Mask')

restored_image = inpaint.inpaint_biharmonic(damaged_image, mask, channel_axis=-1)
show_image(restored_image)

ถ้าส่วนที่เสียไม่ซับซ้อนมาก จะ restore ได้เนียน

In [ ]:
def get_mask(image): # สร้างรอยดำบนภาพก่อน
    mask = np.ones(image.shape[:-1])
    mask[200:300, 15:50] = 0
    return mask

mask = get_mask(defect_image)
damaged_image = defect_image.copy()
damaged_image[mask == 0] = 0
show_image(damaged_image)

In [ ]:
def get_mask(image): # อันนี้เป็น mask ที่จะใช้ restore ภาพ
    ''' Creates mask with four defect regions '''
    mask = np.zeros(image.shape[:-1])
    mask[200:300, 15:50] = 1
    return mask

mask = get_mask(damaged_image)
show_image(mask, title='Mask')

In [ ]:
restored_image = inpaint.inpaint_biharmonic(damaged_image, mask, channel_axis=-1)
show_image(restored_image)

* ***ประเด็นคือ แบบนี้ ต้องรู้ตำแหน่งของจุดที่เสีย เช่น 200:300, 15:50 ให้ชัดเจนสิ***

    * จะทำงั้นได้ ต้องดูขนาดของมิติจาก `image.shape` ก่อนด้วย

# **Apply Noise in Scikit-Image**

In [ ]:
from skimage.util import random_noise

image = data.cat()
noisy_image = random_noise(image)
plot_comparison(image, noisy_image, "Noisy Image")

## **Reducing Noise**

In [ ]:
from skimage.restoration import denoise_tv_chambolle, denoise_bilateral

denoise_image = denoise_tv_chambolle(noisy_image, weight=0.1, channel_axis=-1)
denoise_image_2 = denoise_bilateral(noisy_image, channel_axis=-1)
plot_comparison(noisy_image, denoise_image, "Denoise")
plot_comparison(noisy_image, denoise_image_2, "Denoise_Bilateral")


# **Segmentation**

In [ ]:
from skimage.segmentation import slic
from skimage.color import label2rgb

image = data.coffee()
segments = slic(image, n_segments=300)
segmented_image = label2rgb(segments, image, kind='avg')
show_image(image)
plt.imshow(segmented_image)
plt.show()


# **Contours**

The image is not grayscale or binary yet. This means we need to perform some image pre-processing steps before looking for the contours.

In [ ]:
from skimage.measure import find_contours

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 3/dices.png")
gray = rgb2gray(image[:, :, :3])
plot_comparison(image, gray, "Grayscale")

In [ ]:
# Obtain the optimal threshold value for the image and set it as thresh
thresh = threshold_otsu(gray)

# Apply thresholding to the image once you have the optimal threshold value thresh
thresholded_image = gray > thresh
show_image(thresholded_image.astype(float))

In [ ]:
contours = find_contours(thresholded_image, 0.8)
show_image_contour(gray, contours)

In [ ]:
# Import the modules
from skimage import data, measure

# Obtain the horse image
horse_image = data.horse()

# Find the contours with a constant level value of 0.8
contours = measure.find_contours(horse_image, 0.8)

# Shows the image with contours found
show_image_contour(horse_image, contours)

### **Count the dots in a dice's image**

Now we have found the contours, we can extract information from it.

We'll determine what number was rolled for the dice, by counting the dots in the image.

Create a list with all contour's shapes as shape_contours. You can see all the contours shapes by calling shape_contours in the console, once you have created it.

Check that most of the contours aren't bigger in size than 50. If you count them, they are the exact number of dots in the image.

In [ ]:
# Create list with the shape of each contour
shape_contours = [cnt.shape[0] for cnt in contours]

# Set 50 as the maximum size of the dots shape
max_dots_shape = 50

# Count dots in contours excluding bigger than dots size
dots_contours = [cnt for cnt in contours if np.shape(cnt)[0] < max_dots_shape]

# Shows all contours found
show_image_contour(gray, contours)

# Print the dice's number
print("Dice's dots number: {}. ".format(len(dots_contours)))

# **Edges**

In [ ]:
from skimage.feature import canny

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 4/toronjas.jpg")
gray = rgb2gray(image)

# Apply canny edge detector
edges = canny(gray)

# Show resulting image
plot_comparison(gray, edges, "Edges")

# **Corners**

In [ ]:
from skimage.feature import corner_harris, corner_peaks

image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 2/building_image.jpg")
image = rgb2gray(image)
corner = corner_harris(image)
plot_comparison(image, corner, "Corners")

## **Coordinates of corners**

In [ ]:
coords = corner_peaks(corner_harris(image), min_distance=100, threshold_rel=0.02)

print(f'{len(coords)} corners are detected.')

show_image(image, "Original")
show_image_with_corners(image, coords)

# **Face Detection**

* Need an xml file

In [ ]:
image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 4/face_det25.jpg")
show_image(image)

In [ ]:
from skimage.feature import Cascade
from matplotlib import patches

trained_file = data.lbp_frontal_face_cascade_filename()

# Initialize the detector
detector = Cascade(trained_file)
detected = detector.detect_multi_scale(img=image, scale_factor=1.2, step_ratio=1, min_size=(10, 10), max_size=(200, 200))
print(detected)
show_detected_face(image, detected)


### **Segmentation and face detection**

You learned how to make processes more computationally efficient with unsupervised superpixel segmentation.  Using the **`slic()`** function for segmentation, pre-process the image before passing it to the face detector.

- Apply superpixel segmentation and obtain the segments a.k.a. labels using **`slic()`**.

- Obtain the segmented image using **`label2rgb()`**, passing the `segments` and `profile_image`.

- Detect the faces, using the detector with multi scale method.

In [ ]:
profile_image = plt.imread("/content/Image Processing with Python course exercise dataset/chapter 4/face_det9.jpg")
show_image(profile_image)

In [ ]:
# Obtain the segmentation with default 100 regions
segments = slic(profile_image, n_segments=100)

# Obtain segmented image using label2rgb
segmented_image = label2rgb(segments, profile_image, kind='avg')

# Detect the faces with multi scale method
detected = detector.detect_multi_scale(img=segmented_image,
                                       scale_factor=1.2,
                                       step_ratio=1,
                                       min_size=(10, 10), max_size=(1000, 1000))

# Show the detected faces
show_detected_face(segmented_image, detected)

# **Censoring faces**

In [ ]:
def getFace(d):
    ''' Extracts the face rectangle from the image using the
    coordinates of the detected.'''
    # X and Y starting points of the face rectangle
    x, y = d['r'], d['c']
    # The width and height of the face rectangle
    width, height = d['r'] + d['width'], d['c'] + d['height']
    # Extract the detected face
    face= image[x:width, y:height]
    return face

def getFaceRectangle(d):
    ''' Extracts the face from the image using the coordinates of the detected image '''
    # X and Y starting points of the face rectangle
    x, y  = d['r'], d['c']

    # The width and height of the face rectangle
    width, height = d['r'] + d['width'],  d['c'] + d['height']

    # Extract the detected face
    face= group_image[ x:width, y:height]
    return face

def mergeBlurryFace(original, gaussian_image):
     # X and Y starting points of the face rectangle
    x, y  = d['r'], d['c']
    # The width and height of the face rectangle
    width, height = d['r'] + d['width'],  d['c'] + d['height']

    original[ x:width, y:height] =  gaussian_image
    return original

In [ ]:
from skimage.feature import Cascade
from skimage.filters import gaussian

group_image = plt.imread(
    "/content/Image Processing with Python course exercise dataset/chapter 4/face_det_friends22.jpg"
).copy()
trained_file = data.lbp_frontal_face_cascade_filename()
detector = Cascade(trained_file)
detected = detector.detect_multi_scale(img=group_image,
                                       scale_factor=1.2, step_ratio=1,
                                       min_size=(10, 10), max_size=(100, 100))
# For each detected face
for d in detected:
    # Obtain the face rectangle from detected coordinates
    face = getFace(d)

    # Apply gaussian filter to extracted face
    blurred_face = gaussian(face, channel_axis=-1, sigma=8)

    # Merge this blurry face to our final image and show it
    resulting_image = mergeBlurryFace(group_image, blurred_face)
show_image(resulting_image, "Blurred faces")



### **Restore damaged image**

- Import the necessary module to apply restoration on the image.
- Rotate the image by calling the function **`rotate()`**.
- Use the chambolle algorithm to remove the noise from the image.
- With the mask provided, use the biharmonic method to restore the missing parts of the image and obtain the final image.

In [ ]:
damaged_image = plt.imread("/content/sally_damaged_image.jpg").copy()
show_image(damaged_image)

def get_mask(image):
    # Create mask with three defect regions: left, middle, right respectively
    mask_for_solution = np.zeros(image.shape[:-1])
    mask_for_solution[450:475, 470:495] = 1
    mask_for_solution[320:355, 140:175] = 1
    mask_for_solution[130:155, 345:370] = 1
    return mask_for_solution

# Import the necessary modules
from skimage.restoration import denoise_tv_chambolle, inpaint
from skimage.transform import rotate

# Transform the image so it's not rotated
upright_img = rotate(damaged_image, 20)

# Remove noise from the image, using the chambolle method
upright_img_without_noise = denoise_tv_chambolle(upright_img,weight=0.1, channel_axis=-1)

# Reconstruct the image missing parts
mask = get_mask(upright_img)
result = inpaint.inpaint_biharmonic(upright_img_without_noise, mask, channel_axis=-1)

show_image(result)